# Group Chat Multi-Agent Pattern

In the **group chat pattern**, multiple agents share a **single conversation transcript**. Each turn, a **speaker selection** policy decides who speaks next — round-robin rotation, a manager LLM, or rule-based selector. Agents read the full (or summarized) history and contribute messages until a **termination condition** is met.

```
User: "Draft v2.4.0 release announcement"
  Researcher: "Here are the changelog items..."
  Writer: "Draft: Release v2.4.0 is now available..."
  Critic: "APPROVED — meets style guide"
  Manager: TERMINATE
```

Unlike a **sequential pipeline**, turn order is **not fixed at compile time** — it emerges from the selector (though you can constrain it with rules). Unlike a **supervisor**, agents **speak to the group**, not only to a router.

This notebook implements **ReleaseFlow** group chat in plain Python: transcript model, round-robin and manager selectors, termination policies, transcript budgeting, and comparison demos. No frameworks or prior notebooks are required.

In [ ]:
"""
ReleaseFlow — group chat multi-agent environment.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Protocol

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

RUNBOOKS: list[dict[str, str]] = [
    {"id": "comm-201", "title": "Customer communication standards",
     "text": "Lead with user benefit, list breaking changes separately, include support channel #releases."},
]

STYLE_GUIDE = {
    "must_include": ["version number", "breaking changes section", "support channel"],
    "max_words": 200,
}

print(f"ReleaseFlow corpus: {len(CHANGELOG)} changelog items")

---

## 1. Pattern Definition

| Property | Group chat pattern |
|----------|-------------------|
| **Memory** | Shared message transcript |
| **Routing** | Speaker selector each turn |
| **Visibility** | Agents see prior messages (unless scoped) |
| **Termination** | Max rounds, keyword, or manager decision |
| **UX fit** | Chat interfaces, brainstorming, debate |

```
        ┌─────────────────────────────────┐
        │      Shared Transcript            │
        │  [user, researcher, writer, ...]  │
        └───────────────┬─────────────────┘
                        │
              ┌─────────▼─────────┐
              │  Speaker Selector  │  ← round-robin | manager | rules
              └─────────┬─────────┘
                        │
         ┌──────────────┼──────────────┐
         ▼              ▼              ▼
    Researcher       Writer         Critic
```

---

## 2. Group Chat vs Sequential vs Supervisor

| | **Sequential** | **Group chat** | **Supervisor** |
|--|----------------|----------------|----------------|
| **Next actor** | Fixed order | Selector picks | Router delegates |
| **Communication** | Artifacts in state | Public transcript | Often worker → supervisor |
| **Auditability** | Highest | Medium | High |
| **Token cost** | Lowest | Grows with transcript | Medium |
| **Exploration** | Low | High | Medium |

Use group chat when the **product is a conversation** or agents benefit from seeing each other's reasoning — not when you need a rigid compliance pipeline.

---

## 3. Transcript Model

Every group chat needs a canonical message list and optional **artifacts** (structured outputs extracted from chat).

In [ ]:
@dataclass
class ChatMessage:
    speaker: str
    content: str
    msg_type: str = "text"  # text | system | termination
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def to_dict(self) -> dict[str, str]:
        return {"speaker": self.speaker, "content": self.content, "type": self.msg_type, "ts": self.timestamp}


@dataclass
class GroupChatState:
    session_id: str
    goal: str
    messages: list[ChatMessage] = field(default_factory=list)
    artifacts: dict[str, Any] = field(default_factory=dict)
    round_count: int = 0
    terminated: bool = False
    termination_reason: str = ""

    def add(self, speaker: str, content: str, msg_type: str = "text") -> None:
        self.messages.append(ChatMessage(speaker, content, msg_type))

    def transcript_text(self, max_messages: int | None = None) -> str:
        msgs = self.messages[-max_messages:] if max_messages else self.messages
        return "\n".join(f"{m.speaker}: {m.content}" for m in msgs)

    def token_estimate(self) -> int:
        return len(self.transcript_text().split())


state = GroupChatState(session_id="demo", goal="test")
state.add("user", "Hello team")
print(state.transcript_text())
print(f"Token estimate: {state.token_estimate()} words")

---

## 4. Chat Agents — Read Transcript, Post Reply

Each agent inspects `GroupChatState` and appends a message. They also update `artifacts` for structured downstream use.

In [ ]:
class ChatAgent(Protocol):
    name: str

    def respond(self, state: GroupChatState) -> str:
        ...


class ResearcherChatAgent:
    name = "researcher"

    def respond(self, state: GroupChatState) -> str:
        if "research" in state.artifacts:
            return "Research already shared — see prior message."
        facts = {
            "version": "2.4.0",
            "changes": [c["item"] for c in CHANGELOG],
            "comm_standards": RUNBOOKS[0]["text"],
        }
        state.artifacts["research"] = facts
        return (
            f"Research for v{facts['version']}: {len(facts['changes'])} changes — "
            + "; ".join(facts["changes"][:2]) + "..."
        )


class WriterChatAgent:
    name = "writer"

    def respond(self, state: GroupChatState) -> str:
        facts = state.artifacts.get("research", {})
        if not facts:
            return "Waiting for researcher facts before drafting."
        feedback = ""
        for m in reversed(state.messages):
            if m.speaker == "critic" and "REJECTED" in m.content:
                feedback = m.content
                break
        draft = (
            f"Release v{facts.get('version', '?')} is now available.\n\n"
            f"What's new: " + "; ".join(facts.get("changes", [])) + ".\n\n"
            f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
            f"Questions? Contact #releases."
        )
        if feedback and "too long" in feedback.lower():
            draft = "\n".join(draft.split("\n")[:4]) + "\n\nQuestions? Contact #releases."
        state.artifacts["draft"] = draft
        return f"Draft ({len(draft.split())} words):\n{draft[:200]}..."


class CriticChatAgent:
    name = "critic"

    def respond(self, state: GroupChatState) -> str:
        draft = state.artifacts.get("draft", "")
        if not draft:
            return "No draft to review yet."
        issues = []
        if "v2.4" not in draft:
            issues.append("missing version")
        if "Breaking changes" not in draft:
            issues.append("missing breaking section")
        if "#releases" not in draft:
            issues.append("missing support channel")
        if len(draft.split()) > STYLE_GUIDE["max_words"]:
            issues.append(f"too long ({len(draft.split())} words)")
        if issues:
            state.artifacts["approved"] = False
            return f"REJECTED: {', '.join(issues)}"
        state.artifacts["approved"] = True
        return "APPROVED: draft meets style guide. TERMINATE"


ROSTER: dict[str, ChatAgent] = {
    "researcher": ResearcherChatAgent(),
    "writer": WriterChatAgent(),
    "critic": CriticChatAgent(),
}
print("Chat agents:", list(ROSTER.keys()))

---

## 5. Speaker Selection Strategies

| Strategy | How next speaker is chosen | Predictability |
|----------|---------------------------|----------------|
| **Round-robin** | Rotate roster index each turn | High |
| **Rule-based manager** | If/else on artifacts and transcript | High |
| **LLM manager** | Model reads chat, outputs next name | Low |
| **Priority queue** | Highest-priority agent with pending work | Medium |

In [ ]:
SpeakerSelector = Callable[[GroupChatState, list[str], int], str | None]


def round_robin_selector(state: GroupChatState, roster: list[str], turn: int) -> str | None:
    if state.terminated:
        return None
    return roster[turn % len(roster)]


def rule_based_manager(state: GroupChatState, roster: list[str], turn: int) -> str | None:
    """Deterministic manager — mimics GroupChatManager without an LLM."""
    if state.terminated:
        return None
    if "research" not in state.artifacts:
        return "researcher"
    if "draft" not in state.artifacts:
        return "writer"
    if "approved" not in state.artifacts:
        return "critic"
    if state.artifacts.get("approved"):
        return None
    if state.artifacts.get("revision_count", 0) < 1:
        state.artifacts["revision_count"] = state.artifacts.get("revision_count", 0) + 1
        state.artifacts.pop("draft", None)
        state.artifacts.pop("approved", None)
        return "writer"
    return None


print("Selectors: round_robin_selector, rule_based_manager")

---

## 6. GroupChatEngine — Core Loop

The engine runs: **select speaker → agent responds → check termination → repeat**.

In [ ]:
class TerminationPolicy:
    TERMINATE_KEYWORDS = ("TERMINATE", "APPROVED:")

    def __init__(self, max_rounds: int = 12, max_tokens: int = 4000) -> None:
        self.max_rounds = max_rounds
        self.max_tokens = max_tokens

    def check(self, state: GroupChatState, last_message: str) -> str | None:
        if "TERMINATE" in last_message:
            return "keyword_terminate"
        if state.artifacts.get("approved"):
            return "approved"
        if state.round_count >= self.max_rounds:
            return "max_rounds"
        if state.token_estimate() > self.max_tokens:
            return "token_budget"
        return None


@dataclass
class GroupChatResult:
    session_id: str
    terminated: bool
    reason: str
    rounds: int
    message_count: int
    approved: bool
    speaker_log: list[str]
    token_estimate: int
    draft_preview: str = ""


class GroupChatEngine:
    def __init__(
        self,
        roster: dict[str, ChatAgent],
        selector: SpeakerSelector,
        termination: TerminationPolicy | None = None,
    ) -> None:
        self.roster = roster
        self.roster_names = list(roster.keys())
        self.selector = selector
        self.termination = termination or TerminationPolicy()

    def run(self, goal: str, *, user_proxy: str = "user") -> GroupChatResult:
        state = GroupChatState(session_id=f"chat_{uuid.uuid4().hex[:8]}", goal=goal)
        state.add(user_proxy, goal)
        speaker_log: list[str] = []
        turn = 0

        while True:
            speaker = self.selector(state, self.roster_names, turn)
            if speaker is None:
                state.terminated = True
                state.termination_reason = "selector_done"
                break

            agent = self.roster[speaker]
            reply = agent.respond(state)
            state.add(speaker, reply)
            speaker_log.append(speaker)
            state.round_count += 1
            turn += 1

            reason = self.termination.check(state, reply)
            if reason:
                state.terminated = True
                state.termination_reason = reason
                state.add("manager", f"Session ended: {reason}", msg_type="termination")
                break

        return GroupChatResult(
            session_id=state.session_id,
            terminated=state.terminated,
            reason=state.termination_reason,
            rounds=state.round_count,
            message_count=len(state.messages),
            approved=bool(state.artifacts.get("approved")),
            speaker_log=speaker_log,
            token_estimate=state.token_estimate(),
            draft_preview=state.artifacts.get("draft", "")[:200],
        )


print("GroupChatEngine ready")

---

## 7. Round-Robin Group Chat Demo

Agents speak in rotation: researcher → writer → critic → researcher → ...

Round-robin is simple but can waste turns (e.g. critic speaks before draft exists).

In [ ]:
rr_engine = GroupChatEngine(ROSTER, round_robin_selector, TerminationPolicy(max_rounds=9))
rr_result = rr_engine.run("Produce v2.4.0 customer release announcement")

print(f"Session: {rr_result.session_id}")
print(f"Terminated: {rr_result.terminated} ({rr_result.reason})")
print(f"Rounds: {rr_result.rounds} | Approved: {rr_result.approved}")
print(f"Speaker order: {' → '.join(rr_result.speaker_log)}")
print(f"Token estimate: {rr_result.token_estimate} words")

---

## 8. Manager-Selected Group Chat Demo

A **rule-based manager** skips wasteful turns — closer to production `GroupChatManager` behavior without an LLM selector.

In [ ]:
mgr_engine = GroupChatEngine(ROSTER, rule_based_manager, TerminationPolicy(max_rounds=10))
mgr_result = mgr_engine.run("Produce v2.4.0 customer release announcement")

print(f"Manager-selected session: {mgr_result.session_id}")
print(f"Reason: {mgr_result.reason} | Approved: {mgr_result.approved}")
print(f"Rounds: {mgr_result.rounds} (vs round-robin {rr_result.rounds})")
print(f"Speaker order: {' → '.join(mgr_result.speaker_log)}")

---

## 9. Transcript Replay — Full Conversation

In [ ]:
def run_and_print_transcript(engine: GroupChatEngine, goal: str) -> GroupChatResult:
    """Run chat and display speaker order + outcome."""
    result = engine.run(goal)
    print(f"=== Session {result.session_id} ===")
    print(f"Speakers: {' → '.join(result.speaker_log)}")
    print(f"Ended: {result.reason} | Approved: {result.approved}")
    print(f"Draft preview:\n{result.draft_preview}...")
    return result


run_and_print_transcript(mgr_engine, "Show me the release announcement workflow")

---

## 10. Human Proxy Pattern

A **user proxy** agent represents the human in the chat — initiating tasks and optionally approving termination.

In [ ]:
class HumanProxyAgent:
    name = "human_proxy"

    def __init__(self, auto_approve: bool = True) -> None:
        self.auto_approve = auto_approve

    def respond(self, state: GroupChatState) -> str:
        if state.artifacts.get("approved") and not state.artifacts.get("human_ok"):
            if self.auto_approve:
                state.artifacts["human_ok"] = True
                return "Human approves publication. TERMINATE"
            return "Awaiting human click to publish."
        return "Please continue toward an approved draft."


def human_in_loop_selector(state: GroupChatState, roster: list[str], turn: int) -> str | None:
    base = rule_based_manager(state, [n for n in roster if n != "human_proxy"], turn)
    if base:
        return base
    if state.artifacts.get("approved") and not state.artifacts.get("human_ok"):
        return "human_proxy"
    return None


hitl_roster = {**ROSTER, "human_proxy": HumanProxyAgent(auto_approve=True)}
hitl_engine = GroupChatEngine(hitl_roster, human_in_loop_selector)
hitl_result = hitl_engine.run("v2.4.0 announcement with human gate")
print(f"HITL approved: {hitl_result.approved} | Reason: {hitl_result.reason}")
print(f"Speakers: {' → '.join(hitl_result.speaker_log)}")

---

## 11. Transcript Budgeting

Long group chats blow the context window. Mitigations:

| Technique | Effect |
|-----------|--------|
| **Max rounds** | Hard cap on turns |
| **Token budget** | Stop when transcript exceeds N words |
| **Rolling window** | Agents see last K messages only |
| **Summarize mid-chat** | Compress older messages into one system note |

In [ ]:
def summarize_transcript(messages: list[ChatMessage], keep_last: int = 4) -> str:
    if len(messages) <= keep_last:
        return "\n".join(f"{m.speaker}: {m.content[:80]}" for m in messages)
    older = messages[:-keep_last]
    recent = messages[-keep_last:]
    speakers = list(dict.fromkeys(m.speaker for m in older))
    summary = f"[Earlier: {len(older)} messages from {', '.join(speakers)}]"
    recent_text = "\n".join(f"{m.speaker}: {m.content[:120]}" for m in recent)
    return summary + "\n" + recent_text


sample_msgs = [
    ChatMessage("user", "Start"),
    ChatMessage("researcher", "Facts..."),
    ChatMessage("writer", "Draft v1..."),
    ChatMessage("critic", "REJECTED"),
    ChatMessage("writer", "Draft v2..."),
    ChatMessage("critic", "APPROVED TERMINATE"),
]
print(summarize_transcript(sample_msgs, keep_last=3))

---

## 12. Side-by-Side — Selector Comparison

In [ ]:
GOAL = "Produce v2.4.0 customer release announcement"
COMPARISON: list[dict[str, Any]] = []

for label, selector in [("round_robin", round_robin_selector), ("manager", rule_based_manager)]:
    eng = GroupChatEngine(ROSTER, selector, TerminationPolicy(max_rounds=12))
    r = eng.run(GOAL)
    COMPARISON.append({
        "selector": label,
        "rounds": r.rounds,
        "approved": r.approved,
        "tokens": r.token_estimate,
        "reason": r.reason,
    })

print(f"{'Selector':<14} {'Rounds':<8} {'Approved':<10} {'Tokens':<8} Reason")
print("-" * 55)
for row in COMPARISON:
    print(f"{row['selector']:<14} {row['rounds']:<8} {str(row['approved']):<10} {row['tokens']:<8} {row['reason']}")

---

## 13. Structured Trace for Group Chat

In [ ]:
def result_to_trace_json(result: GroupChatResult) -> str:
    events = [
        {"event": "session_start", "session_id": result.session_id},
    ]
    for i, speaker in enumerate(result.speaker_log, start=1):
        events.append({"event": "speaker", "turn": i, "name": speaker})
    events.append({
        "event": "session_end",
        "reason": result.reason,
        "approved": result.approved,
        "rounds": result.rounds,
    })
    return "\n".join(json.dumps(e) for e in events)


print(result_to_trace_json(mgr_result))

---

## 14. When to Use Group Chat

| Good fit | Poor fit |
|----------|----------|
| Brainstorming, debate, research sessions | Fixed compliance pipeline |
| Chat UI products | Headless batch ETL |
| Human-in-the-loop coding assistants | Strict step-order audits |
| Exploring unknown solution paths | Known A→B→C every time |

**Production tip:** Even in chat UX, use a **manager with rules** or **max_rounds** — unconstrained emergent chat is expensive and may not terminate.

---

## 15. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| Unbounded rounds | Infinite loop, cost explosion | `max_rounds` + termination policy |
| Full transcript every call | Token cost | Rolling window or summarize |
| Round-robin for pipelines | Wasted turns | Manager selector or sequential |
| No artifacts | Only chat text — hard to publish | Extract `draft`, `approved` to state |
| Agents without role boundaries | All agents do everything | Narrow system prompts + tools |

---

## 16. Optional Live LLM Manager

Replace `rule_based_manager` with an LLM that outputs the next speaker name.

In [ ]:
USE_LIVE_LLM = False


def llm_manager_selector(state: GroupChatState, roster: list[str], turn: int) -> str | None:
    if not USE_LIVE_LLM:
        return rule_based_manager(state, roster, turn)
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        prompt = (
            f"Goal: {state.goal}\nTranscript:\n{state.transcript_text(max_messages=8)}\n"
            f"Roster: {roster}\nReply with ONE name from roster or DONE."
        )
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        choice = (resp.choices[0].message.content or "").strip().lower()
        if "done" in choice:
            return None
        for name in roster:
            if name in choice:
                return name
        return rule_based_manager(state, roster, turn)
    except Exception:
        return rule_based_manager(state, roster, turn)


if USE_LIVE_LLM:
    live = GroupChatEngine(ROSTER, llm_manager_selector).run(GOAL)
    print(live.speaker_log)
else:
    print("Offline mode — manager uses rule_based_manager.")
    print("Conceptual LLM manager prompt: transcript + roster → next speaker or DONE")

---

## 17. Quiz

1. What is the shared data structure in group chat multi-agent systems?
2. How does round-robin differ from a manager selector?
3. Name two termination conditions for a group chat.
4. Why extract artifacts alongside the transcript?
5. When should you prefer sequential over group chat?

---

## 18. Summary

| Concept | Takeaway |
|---------|----------|
| Group chat | Shared transcript + speaker selection each turn |
| Round-robin | Simple, predictable, can waste turns |
| Manager selector | Skips irrelevant speakers; closer to production |
| Termination | `max_rounds`, keywords, `approved` artifact |
| Human proxy | Human gate before final TERMINATE |
| Token budget | Summarize or window long transcripts |
| Artifacts | Structured `draft` / `approved` alongside chat text |

Group chat shines when **conversation is the product**. For release pipelines with fixed stages, pair chat UX with a **rule-based manager** or use a **sequential** orchestrator under the hood.